In [1]:
from pathlib import Path
import pandas as pd

# 1. Đường dẫn
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name.lower() in ("notebook", "notebooks"):
    PROJECT_DIR = PROJECT_DIR.parent

GEN_DIR = PROJECT_DIR / "results/llm/reproduction"
EVAL_DIR = PROJECT_DIR / "results/evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Tìm tất cả các phiên thực nghiệm
run_files = sorted(GEN_DIR.glob("generation_run_*.csv"))
assert run_files, "Không tìm thấy kết quả thực nghiệm"

# 3. Kiểm tra từng phiên
expected_pairs = {
    (qid, method)
    for qid in range(1, 44)
    for method in ("SQL", "SPARQL")
}

for file in run_files:
    df = pd.read_csv(file, keep_default_na=False)

    pairs = set(zip(df["ID"], df["Method"]))

    assert len(df) == 86, f"Sai số lượng: {file.name}"
    assert pairs == expected_pairs, f"Sai danh sách câu hỏi: {file.name}"
    assert df["Status"].eq("GENERATED").all(), file.name
    assert df["Generated Query"].astype(str).str.strip().ne("").all()

    print(file.name, "- OK")

# 4. Tổng hợp
N_RUNS = len(run_files)

print("\n=== GPT-5.4 EXPERIMENTS ===")
print("Total runs:", N_RUNS)
print("Queries per run:", 86)
print("Total generated queries:", N_RUNS * 86)
print("Validation: OK")


generation_run_01.csv - OK
generation_run_02.csv - OK
generation_run_03.csv - OK
generation_run_04.csv - OK
generation_run_05.csv - OK

=== GPT-5.4 EXPERIMENTS ===
Total runs: 5
Queries per run: 86
Total generated queries: 430
Validation: OK


In [2]:
import duckdb
import time

DB_FILE = PROJECT_DIR / "results/sql/acme.duckdb"
SQL_DIR = EVAL_DIR / "sql_runs"
SQL_DIR.mkdir(parents=True, exist_ok=True)

assert DB_FILE.exists(), "Không tìm thấy database"

for file in run_files:
    run_id = file.stem.split("_")[-1]
    run_dir = SQL_DIR / f"run_{run_id}"
    run_dir.mkdir(exist_ok=True)

    df = pd.read_csv(file, keep_default_na=False)
    sql_df = df[df["Method"] == "SQL"].sort_values("ID")

    log_file = run_dir / "execution_log.csv"
    logs = []

    con = duckdb.connect(str(DB_FILE), read_only=True)

    for _, row in sql_df.iterrows():
        qid = int(row["ID"])
        query = row["Generated Query"].strip()
        output = run_dir / f"{qid:02d}.csv"

        start = time.perf_counter()
        status = "OK"
        error = ""

        # Chỉ thực thi truy vấn đọc dữ liệu
        try:
            if not query.upper().startswith(("SELECT", "WITH")):
                raise ValueError("Not a SELECT/WITH query")

            result = con.execute(query).fetchdf()
            result.to_csv(output, index=False)

        except Exception as e:
            status = "ERROR"
            error = str(e)
            output.unlink(missing_ok=True)

        logs.append({
            "ID": qid,
            "Status": status,
            "Error": error,
            "Time": round(time.perf_counter() - start, 3)
        })

        # Lưu tiến độ sau mỗi truy vấn
        pd.DataFrame(logs).to_csv(log_file, index=False)

    con.close()

    print(f"Run {run_id}:", pd.DataFrame(logs)["Status"].value_counts().to_dict())

print("SQL execution completed.")


Run 01: {'OK': 42, 'ERROR': 1}
Run 02: {'OK': 42, 'ERROR': 1}
Run 03: {'OK': 40, 'ERROR': 3}
Run 04: {'OK': 42, 'ERROR': 1}
Run 05: {'OK': 40, 'ERROR': 3}
SQL execution completed.


In [3]:
import subprocess
import re

# 1. Cấu hình Ontop
ONTOP = Path.home() / "tools/ontop/ontop.bat"
MAPPING = PROJECT_DIR / "results/ontop/acme_mapping.ttl"
ONTOLOGY = PROJECT_DIR / "data/ACME_Insurance/ontology/insurance.ttl"
PROPERTIES = PROJECT_DIR / "results/ontop/duckdb.properties"

SPARQL_DIR = EVAL_DIR / "sparql_runs"
SPARQL_DIR.mkdir(parents=True, exist_ok=True)

for path in [ONTOP, MAPPING, ONTOLOGY, PROPERTIES]:
    assert path.exists(), f"Missing: {path}"

# 2. Thực thi từng run
for file in run_files:
    run_id = file.stem.split("_")[-1]
    run_dir = SPARQL_DIR / f"run_{run_id}"
    run_dir.mkdir(exist_ok=True)

    queries_dir = run_dir / "queries"
    queries_dir.mkdir(exist_ok=True)

    df = pd.read_csv(file, keep_default_na=False)
    queries = df[df["Method"] == "SPARQL"].sort_values("ID")

    log_file = run_dir / "execution_log.csv"
    records = {}

    if log_file.exists():
        old = pd.read_csv(log_file, keep_default_na=False)
        records = {int(r["ID"]): r.to_dict() for _, r in old.iterrows()}

    for _, row in queries.iterrows():
        qid = int(row["ID"])
        output = run_dir / f"{qid:02d}.csv"

        # Bỏ qua câu đã xử lý
        if qid in records and (
            records[qid]["Status"] != "OK" or output.exists()
        ):
            continue

        # Giữ nguyên SPARQL gốc
        raw_query = row["Generated Query"].strip()
        (queries_dir / f"{qid:02d}_original.rq").write_text(
            raw_query, encoding="utf-8"
        )

        # Ontop local không sử dụng SERVICE <mapped>
        local_query = re.sub(
            r"SERVICE\s*<mapped>\s*(?=\{)",
            "", raw_query, flags=re.IGNORECASE
        )

        query_file = queries_dir / f"{qid:02d}_ontop.rq"
        query_file.write_text(local_query, encoding="utf-8")

        # Thực thi với Ontop
        status = "OK"
        error = ""
        output.unlink(missing_ok=True)

        try:
            process = subprocess.run(
                [
                    str(ONTOP), "query",
                    "-m", MAPPING.resolve().as_uri(),
                    "-t", str(ONTOLOGY.resolve()),
                    "-p", PROPERTIES.resolve().as_uri(),
                    "-q", str(query_file.resolve()),
                    "-o", str(output.resolve())
                ],
                cwd=ONTOP.parent,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=60
            )

            if process.returncode != 0 or not output.exists():
                raise RuntimeError(process.stderr or process.stdout)

        except subprocess.TimeoutExpired:
            status = "TIMEOUT"
            error = "Exceeded 60 seconds"

        except Exception as e:
            status = "ERROR"
            error = str(e)[-500:]

        if status != "OK":
            output.unlink(missing_ok=True)

        records[qid] = {
            "ID": qid,
            "Status": status,
            "Error": error
        }

        pd.DataFrame(
            sorted(records.values(), key=lambda r: int(r["ID"]))
        ).to_csv(log_file, index=False)

    log = pd.read_csv(log_file)
    assert len(log) == 43
    print(f"Run {run_id}:", log["Status"].value_counts().to_dict())

print("SPARQL execution completed.")

Run 01: {'OK': 36, 'ERROR': 7}
Run 02: {'OK': 37, 'ERROR': 6}
Run 03: {'OK': 35, 'ERROR': 8}
Run 04: {'OK': 39, 'ERROR': 4}
Run 05: {'OK': 37, 'ERROR': 6}
SPARQL execution completed.


In [4]:
# 1. Thư mục Reference
SQL_REF_DIR = PROJECT_DIR / "results/sql/reference_results"
ONTOP_DIR = PROJECT_DIR / "results/ontop"
SPARQL_REF_DIR = ONTOP_DIR / "reference_results"

for name, folder in [
    ("SQL", SQL_REF_DIR),
    ("SPARQL", SPARQL_REF_DIR)
]:
    files = sorted(folder.glob("*.csv"))
    print(f"\n{name} Reference: {len(files)} files")
    print("Examples:", [f.name for f in files[:10]])

# 2. Đọc bảng ánh xạ Reference
comparison_file = ONTOP_DIR / "reference_comparison.csv"
assert comparison_file.exists()

reference = pd.read_csv(comparison_file, keep_default_na=False)

print("\n=== REFERENCE MAPPING ===")
print("Columns:", reference.columns.tolist())
print("Rows:", len(reference))
display(reference.head(5))

# 3. Kiểm tra ngoại lệ câu 18
print("\nQuestion 18 Reference:")
print("18_ontop.csv exists:",
      (SPARQL_REF_DIR / "18_ontop.csv").exists())



SQL Reference: 45 files
Examples: ['query-08e96beb-2717-4045-a830-43749d3cd18c.csv', 'query-09478b45-7383-4608-b696-26f2f6bbe383.csv', 'query-0a848d91-fc89-4774-ae38-3b1015a1dc42.csv', 'query-244778fe-5504-4639-b174-37f013c52700.csv', 'query-366dfab8-1aea-4d48-ac19-84b98432b89d.csv', 'query-37e3d4bb-7132-47a6-a008-47a37cdb9d2c.csv', 'query-4080805b-3146-45f9-8325-51e2d3c00e8d.csv', 'query-40c8a2e3-8530-4b48-8b03-6364070baa32.csv', 'query-425d1228-4470-4470-a85e-26d5003a1990.csv', 'query-453d5cbf-8fa3-46b2-83d5-820fac1a583c.csv']

SPARQL Reference: 43 files
Examples: ['01.csv', '02.csv', '03.csv', '04.csv', '05.csv', '06.csv', '07.csv', '08.csv', '09.csv', '10.csv']

=== REFERENCE MAPPING ===
Columns: ['ID', 'Status', 'SQL Reference', 'SPARQL Rows']
Rows: 43


,ID,Status,SQL Reference,SPARQL Rows
0,1,MATCH,https://templates.data.world/query-63dd7451-59...,1
1,2,MATCH,https://templates.data.world/query-5e26c413-d1...,2
2,3,MATCH,https://templates.data.world/query-47603eb6-8f...,2
3,4,MATCH,https://templates.data.world/query-37e3d4bb-71...,1
4,5,MATCH,https://templates.data.world/query-425d1228-44...,2



Question 18 Reference:
18_ontop.csv exists: True


In [5]:
from collections import Counter
from decimal import Decimal
from itertools import permutations
import re

# 1. Chuẩn hóa giá trị trước khi so sánh
def normalize(value):
    value = str(value).strip()

    if value == "":
        return ""

    # 20.50 và 20.5 được coi là bằng nhau
    try:
        if re.fullmatch(r"[+-]?\d+(\.\d+)?([eE][+-]?\d+)?", value):
            return str(Decimal(value).normalize())
    except Exception:
        pass

    return value


# 2. So sánh hai bảng kết quả
def same_result(expected_file, actual_file):
    a = pd.read_csv(expected_file, dtype=str, keep_default_na=False)
    b = pd.read_csv(actual_file, dtype=str, keep_default_na=False)

    if a.shape != b.shape:
        return False

    a_rows = [tuple(map(normalize, row)) for row in a.itertuples(index=False, name=None)]
    b_rows = [tuple(map(normalize, row)) for row in b.itertuples(index=False, name=None)]

    target = Counter(a_rows)
    n_cols = a.shape[1]

    # Chỉ thử các cách đổi cột có cùng tập giá trị
    candidates = []
    for i in range(n_cols):
        values_a = Counter(row[i] for row in a_rows)
        candidates.append([
            j for j in range(n_cols)
            if values_a == Counter(row[j] for row in b_rows)
        ])

    if any(not options for options in candidates):
        return False

    order = sorted(range(n_cols), key=lambda i: len(candidates[i]))
    mapping = [None] * n_cols
    used = set()

    def check(k):
        if k == n_cols:
            rows = Counter(
                tuple(row[mapping[i]] for i in range(n_cols))
                for row in b_rows
            )
            return rows == target

        i = order[k]
        for j in candidates[i]:
            if j not in used:
                mapping[i] = j
                used.add(j)

                if check(k + 1):
                    return True

                used.remove(j)

        return False

    return check(0)


# 3. Ánh xạ SQL Reference theo ID câu hỏi
sql_refs = {}

for _, row in reference.iterrows():
    qid = int(row["ID"])
    filename = row["SQL Reference"].rstrip("/").split("/")[-1] + ".csv"
    sql_refs[qid] = SQL_REF_DIR / filename


# 4. Chấm toàn bộ 5 run
records = []

for file in run_files:
    run_id = int(file.stem.split("_")[-1])

    for method in ["SQL", "SPARQL"]:
        folder = EVAL_DIR / f"{method.lower()}_runs" / f"run_{run_id:02d}"
        log = pd.read_csv(folder / "execution_log.csv").set_index("ID")

        assert len(log) == 43

        for qid in range(1, 44):
            if method == "SQL":
                gold = sql_refs[qid]
            else:
                name = "18_ontop.csv" if qid == 18 else f"{qid:02d}.csv"
                gold = SPARQL_REF_DIR / name

            actual = folder / f"{qid:02d}.csv"
            status = log.loc[qid, "Status"]

            if not gold.exists():
                raise FileNotFoundError(gold)

            correct = (
                status == "OK"
                and actual.exists()
                and same_result(gold, actual)
            )

            records.append({
                "Run": run_id,
                "ID": qid,
                "Method": method,
                "Status": status,
                "Correct": int(correct)
            })

# 5. Lưu kết quả đánh giá
ea_df = pd.DataFrame(records)
assert len(ea_df) == 430

ea_df.to_csv(EVAL_DIR / "execution_accuracy_all_runs.csv", index=False)

summary = ea_df.groupby(["Run", "Method"])["Correct"].agg(
    Correct="sum",
    EA="mean"
)

summary["EA"] = (summary["EA"] * 100).round(2)

print("=== EXECUTION ACCURACY (%) ===")
display(summary)

print("Total evaluated:", len(ea_df))
print("Saved: execution_accuracy_all_runs.csv")


=== EXECUTION ACCURACY (%) ===


Correct     EA
Run Method                
1   SPARQL       26  60.47
    SQL          18  41.86
2   SPARQL       26  60.47
    SQL          18  41.86
3   SPARQL       25  58.14
    SQL          19  44.19
4   SPARQL       29  67.44
    SQL          19  44.19
5   SPARQL       27  62.79
    SQL          19  44.19

Total evaluated: 430
Saved: execution_accuracy_all_runs.csv


In [8]:
# 1. Kiểm tra đủ 5 run cho mỗi câu hỏi
counts = ea_df.groupby(["ID", "Method"]).size()
assert (counts == N_RUNS).all()

# 2. OEA: trung bình Correct của từng câu hỏi
oea_df = (
    ea_df.groupby(["ID", "Method"], as_index=False)["Correct"]
    .mean()
    .rename(columns={"Correct": "OEA"})
)

# 3. AOEA: trung bình OEA của 43 câu hỏi
aoea_df = (
    oea_df.groupby("Method", as_index=False)["OEA"]
    .mean()
    .rename(columns={"OEA": "AOEA"})
)

# 4. Lưu kết quả (giữ giá trị 0-1)
oea_df.to_csv(EVAL_DIR / "oea_per_question.csv", index=False)
aoea_df.to_csv(EVAL_DIR / "aoea_summary.csv", index=False)

# 5. Hiển thị dạng phần trăm
print("=== AOEA (%) ===")
display(
    aoea_df.assign(AOEA=lambda x: (x["AOEA"] * 100).round(2))
)

print("\n=== OEA PER QUESTION (%) ===")
display(
    oea_df.assign(OEA=lambda x: (x["OEA"] * 100).round(2)).head(43)
)

print("\nSaved: oea_per_question.csv, aoea_summary.csv")

=== AOEA (%) ===


,Method,AOEA
0,SPARQL,61.86
1,SQL,43.26



=== OEA PER QUESTION (%) ===


,ID,Method,OEA
0,1,SPARQL,0.0
1,1,SQL,0.0
2,2,SPARQL,100.0
3,2,SQL,100.0
4,3,SPARQL,0.0
5,3,SQL,0.0
6,4,SPARQL,60.0
7,4,SQL,0.0
8,5,SPARQL,100.0
9,5,SQL,0.0



Saved: oea_per_question.csv, aoea_summary.csv


In [7]:
from rdflib import Graph, Namespace, RDF
from rdflib.namespace import DCTERMS
import re

# 1. Đọc benchmark gốc
benchmark = Graph()
benchmark.parse(
    PROJECT_DIR / "data/ACME_Insurance/investigation/acme-benchmark.ttl",
    format="turtle"
)

QA = Namespace("http://models.data.world/benchmarks/QandA#")

# 2. Lấy Quadrant theo đúng thứ tự ID câu hỏi
rows = []

for inquiry in sorted(benchmark.subjects(RDF.type, QA.Inquiry), key=str):
    question = str(benchmark.value(inquiry, QA.prompt))

    if question == "Return all the claims by claim number that were closed in 2019":
        continue

    labels = set()

    for ref in benchmark.objects(inquiry, QA.expects):
        title = str(benchmark.value(ref, DCTERMS.title) or "")
        match = re.match(r"(LQLS|HQLS|LQHS|HQHS)\b", title.strip(), re.I)

        if match:
            labels.add(match.group(1).upper())

    assert len(labels) == 1, f"Quadrant không xác định: {question}"

    rows.append({
        "ID": len(rows) + 1,
        "Quadrant": labels.pop()
    })

quadrants = pd.DataFrame(rows)
assert len(quadrants) == 43

print("=== QUESTION DISTRIBUTION ===")
print(quadrants["Quadrant"].value_counts().sort_index())

# 3. Ghép Quadrant với OEA
oea_by_quadrant = oea_df.merge(
    quadrants, on="ID", validate="many_to_one"
)

# 4. Tính AOEA cho từng Quadrant
quadrant_aoea = (
    oea_by_quadrant.groupby(["Quadrant", "Method"])["OEA"]
    .mean()
    .reset_index(name="AOEA")
)

# 5. Lưu kết quả
oea_by_quadrant.to_csv(
    EVAL_DIR / "oea_by_quadrant.csv", index=False
)

quadrant_aoea.to_csv(
    EVAL_DIR / "aoea_by_quadrant.csv", index=False
)

print("\n=== AOEA BY QUADRANT (%) ===")
display(
    quadrant_aoea.assign(AOEA=lambda x: (x["AOEA"] * 100).round(2))
)


=== QUESTION DISTRIBUTION ===
Quadrant
HQHS    10
HQLS    11
LQHS    10
LQLS    12
Name: count, dtype: int64

=== AOEA BY QUADRANT (%) ===


,Quadrant,Method,AOEA
0,HQHS,SPARQL,40.00
1,HQHS,SQL,44.00
2,HQLS,SPARQL,60.00
3,HQLS,SQL,52.73
4,LQHS,SPARQL,60.00
5,LQHS,SQL,10.00
6,LQLS,SPARQL,83.33
7,LQLS,SQL,61.67


In [9]:
# 1. Lấy 43 câu hỏi từ benchmark gốc
benchmark_questions = []

for inquiry in sorted(benchmark.subjects(RDF.type, QA.Inquiry), key=str):
    question = str(benchmark.value(inquiry, QA.prompt))

    if question == "Return all the claims by claim number that were closed in 2019":
        continue

    benchmark_questions.append(question)

# 2. Đọc câu hỏi đã dùng để sinh SQL/SPARQL
generated = pd.read_csv(run_files[0], keep_default_na=False)

question_col = next(
    (c for c in generated.columns
     if c.lower().replace("_", " ").strip() in
     ("question", "question text", "natural language question")),
    None
)

print("Generation columns:", generated.columns.tolist())

# 3. Đối chiếu theo ID
if question_col is None:
    print("Chưa thể xác minh: file generation không có cột Question.")
    print("Cần kiểm tra nguồn gán ID trong Notebook 04.")
else:
    sql_questions = (
        generated[generated["Method"] == "SQL"]
        .sort_values("ID")
    )

    assert sql_questions["ID"].tolist() == list(range(1, 44))

    def clean(text):
        return " ".join(str(text).split()).casefold()

    matches = [
        clean(a) == clean(b)
        for a, b in zip(
            sql_questions[question_col],
            benchmark_questions
        )
    ]

    print(f"Matched: {sum(matches)}/43")
    print("Mismatched IDs:",
          [i + 1 for i, ok in enumerate(matches) if not ok])

    if all(matches):
        print("VALIDATION: OK")
    else:
        print("VALIDATION: FAILED — kiểm tra ID trước khi dùng kết quả Quadrant")


Generation columns: ['ID', 'Inquiry URI', 'Question', 'Method', 'Generated Query', 'Prompt Tokens', 'Output Tokens', 'Time (s)', 'Status', 'Error']
Matched: 43/43
Mismatched IDs: []
VALIDATION: OK
